# Algorithm Comparison: Proposed vs Zheng (TCE 2025) vs Wu (TCCN 2025)
### UAV-Assisted Federated Learning for Smart-Farm Crop Health
**Metrics compared:** E_tx · E_rx · E_total · T_trans · Total Cost · K_reward · FL Accuracy · Matched CHs  
**Case 1:** UAVs = 7, Sensors = 200 → 1 000  
**Case 2:** UAVs = 15, Sensors = 200 → 2 000  
All three algorithms run on *identical* sensor counts, UAV counts, and random seeds.


## Cell 1 — Shared Imports & Physics Parameters

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
import random, time, warnings
warnings.filterwarnings('ignore')

# ── Shared physics (identical for all three algorithms) ───────
AREA_SIZE  = 200
E_cir      = 50e-9
theta      = 10e-12
phi        = 0.0013e-12
D          = np.sqrt(theta / phi)
B          = 20e6
p_h        = 0.2
sigma      = 1e-9
gamma      = 1
H_alt      = 100
g_hu       = 1
C_PD       = 9.26e-4
C_RD       = 2250.0
V_u        = 10.0
v_amp      = 2.0
p_cir      = 0.1
E_MAX      = 10800.0
f_hu       = 2e9
epsilon_u  = 1e-26
alpha_u    = 0.5
beta_u     = 0.5
S_grad     = 1e6
b_u        = 20e6
SNR_u      = 20.0
P_com      = 0.1
W_model    = 1e6
NUM_ROUNDS   = 5
LOCAL_EPOCHS = 3
LR           = 1e-3
TEST_SIZE    = 0.2
g_u_reward   = 1.0
pi_u         = 1e-3
lambda_energy = 1.0
lambda_time   = 0.1
TOTAL_CLIENTS = 5

# ── Wu-specific constants ─────────────────────────────────────
P_HOVER     = 1.0
T_HOVER     = 300.0
E_AAV       = 100.0
DELTA_WU    = 10.0
EPS_EDGE    = 0.1
THETA_LOCAL = 0.5
I_EPS_THETA = DELTA_WU * np.log(1/EPS_EDGE) / (1 - THETA_LOCAL)
B_WU        = 2e6
H0_WU       = 10**(-10/10)
N0_WU       = 10**(-90/10) * 1e-3
P_UE        = 1.0
E_AAV_M     = P_HOVER * T_HOVER + E_AAV

# ── Propulsion power ──────────────────────────────────────────
P_u_prop = C_PD * V_u**3 + C_RD / V_u

np.random.seed(42); random.seed(42)
print("Shared parameters loaded.")
print(f"  P_u = {P_u_prop:.1f} W | D = {D:.1f} m | E_AAV_M = {E_AAV_M:.1f} J")
print(f"  I_eps_theta (Wu) = {I_EPS_THETA:.2f}")


## Cell 2 — Load / Generate Dataset

In [ ]:
try:
    df_full = pd.read_csv("agriculture_dataset.csv")
    print(f"Dataset loaded: {df_full.shape[0]:,} rows x {df_full.shape[1]} cols")
except FileNotFoundError:
    print("agriculture_dataset.csv not found — generating 3000-row synthetic dataset")
    N = 3000
    rng0 = np.random.default_rng(42)
    df_full = pd.DataFrame({
        'High_Resolution_RGB'      : rng0.integers(0,2,N),
        'Multispectral_Images'     : rng0.integers(0,2,N),
        'Thermal_Images'           : rng0.integers(0,2,N),
        'Temporal_Images'          : rng0.integers(0,2,N),
        'Spatial_Resolution'       : rng0.uniform(0.01,2.7,N),
        'GPS_Coordinates'          : rng0.integers(100000,999999,N),
        'Field_Boundaries'         : rng0.integers(1,4,N),
        'Elevation_Data'           : rng0.uniform(10,100,N),
        'Canopy_Coverage'          : rng0.uniform(0,300,N),
        'NDVI'                     : rng0.uniform(0.05,1.05,N),
        'SAVI'                     : rng0.uniform(0.05,0.75,N),
        'Chlorophyll_Content'      : rng0.uniform(0.05,4.2,N),
        'Leaf_Area_Index'          : rng0.uniform(0.05,4.8,N),
        'Crop_Stress_Indicator'    : rng0.integers(0,100,N),
        'Temperature'              : rng0.uniform(10,40,N),
        'Humidity'                 : rng0.uniform(20,95,N),
        'Rainfall'                 : rng0.uniform(0,130,N),
        'Wind_Speed'               : rng0.uniform(0.03,7.5,N),
        'Soil_Moisture'            : rng0.uniform(2,38,N),
        'Soil_pH'                  : rng0.uniform(4.5,8.0,N),
        'Organic_Matter'           : rng0.uniform(0.001,20,N),
        'Pest_Hotspots'            : rng0.integers(0,2,N),
        'Weed_Coverage'            : rng0.uniform(0,9,N),
        'Pest_Damage'              : rng0.integers(0,100,N),
        'Crop_Growth_Stage'        : rng0.integers(1,5,N),
        'Expected_Yield'           : rng0.uniform(900,5200,N),
        'Crop_Type'                : rng0.choice(['Wheat','Maize','Rice','Soybean'],N),
        'Ground_Truth_Segmentation': rng0.integers(0,2,N),
        'Bounding_Boxes'           : rng0.integers(0,10,N),
        'Water_Flow'               : rng0.uniform(0,50,N),
        'Drainage_Features'        : rng0.integers(0,2,N),
    })
    lbl = np.zeros(N, dtype=int)
    for i in range(N):
        s = 0
        if 18 <= df_full.loc[i,'Temperature'] <= 34: s += 1
        if 5  <= df_full.loc[i,'Soil_pH']     <= 7.5: s += 1
        if df_full.loc[i,'Soil_Moisture']      >= 15:  s += 1
        if df_full.loc[i,'NDVI']               >= 0.4: s += 1
        if df_full.loc[i,'Crop_Stress_Indicator'] < 50: s += 1
        lbl[i] = 1 if s >= 3 else 0
    df_full['Crop_Health_Label'] = lbl

target_col   = 'Crop_Health_Label'
drop_cols    = ['Crop_Type','Ground_Truth_Segmentation','Bounding_Boxes']
feature_cols = [c for c in df_full.columns if c not in drop_cols + [target_col]]
df_full[feature_cols] = df_full[feature_cols].apply(pd.to_numeric, errors='coerce')
df_full[target_col]   = pd.to_numeric(df_full[target_col], errors='coerce')
imputer   = SimpleImputer(strategy='mean')
X_all_raw = imputer.fit_transform(df_full[feature_cols])
y_all     = df_full[target_col].fillna(0).values.astype(int)
scaler    = StandardScaler()
X_all     = scaler.fit_transform(X_all_raw)
N_FEATURES = X_all.shape[1]
print(f"Features: {N_FEATURES} | Samples: {len(y_all)} | "
      f"Healthy: {np.sum(y_all==1)} | Unhealthy: {np.sum(y_all==0)}")


## Cell 3 — Shared Energy Helpers + All Three run_one Functions

In [ ]:
# ── Energy helpers shared by all algorithms ──────────────────
def _Etx(k, d):
    return k * (E_cir + (theta*d**2 if d < D else phi*d**4))

def _Erx(k):
    return k * E_cir

def _R(chx, chy, ux, uy):
    d3 = np.sqrt((chx-ux)**2 + (chy-uy)**2 + H_alt**2)
    return B * np.log2(1 + (p_h*g_hu) / (d3**2 * sigma))

def _Ech(Q, R):
    return p_h * Q / max(R, 1e-9)

def _T(Q, R):
    return Q / max(R, 1e-9)

def _Etve(dist):
    return P_u_prop * dist / V_u

def _Ecmp(Q):
    return epsilon_u * f_hu**2 * Q * alpha_u * W_model

def _Ecom_per_uav():
    R_up  = b_u * np.log2(1 + 10**(SNR_u/10))
    T_com = S_grad * alpha_u * beta_u / R_up
    return T_com * P_com

def _wu_latency(Q, chx, chy, ux, uy):
    d = np.sqrt((chx-ux)**2 + (chy-uy)**2)
    snr = (H0_WU*P_UE) / max((d**2+H_alt**2)*N0_WU, 1e-30)
    r   = B_WU * np.log2(1+snr)
    return Q / max(r, 1.0)

E_COM = _Ecom_per_uav()

def _fl_acc(nm, K_ch, rng):
    cov = nm / max(K_ch, 1)
    ns  = max(50, int(len(X_all)*cov))
    idx = rng.choice(len(X_all), size=min(ns,len(X_all)), replace=False)
    Xs  = X_all[idx]; ys = y_all[idx]
    if len(np.unique(ys)) < 2: return 0.5
    Xtr,Xte,ytr,yte = train_test_split(Xs, ys, test_size=TEST_SIZE, random_state=0)
    if len(np.unique(ytr)) < 2: return float(np.mean(yte==ytr[0]))
    m = SGDClassifier(loss='log_loss', max_iter=NUM_ROUNDS*LOCAL_EPOCHS,
                      learning_rate='constant', eta0=LR, random_state=0)
    m.fit(Xtr, ytr)
    return accuracy_score(yte, m.predict(Xte))

def _K_reward(M_u_l, Q_arr, E_rem, n_uavs, acc):
    T_com = S_grad*alpha_u*beta_u / (b_u*np.log2(1+10**(SNR_u/10)))
    K = 0.0
    for u in range(n_uavs):
        if not M_u_l[u]: continue
        Et = E_MAX - E_rem[u]
        Ec = sum(_Ecmp(Q_arr[h]) for h in M_u_l[u])
        Eo = T_com * P_com
        K += g_u_reward*acc - pi_u*(Et+Ec+Eo)
    return K

def _setup(n_sensors, K_ch, seed):
    rng = np.random.RandomState(seed)
    sx  = rng.uniform(0, AREA_SIZE, n_sensors)
    sy  = rng.uniform(0, AREA_SIZE, n_sensors)
    kn  = rng.randint(2000, 6000, n_sensors)
    km  = KMeans(n_clusters=K_ch, init='k-means++', random_state=seed, n_init=10)
    lbl = km.fit_predict(np.column_stack([sx, sy]))
    CHx = np.zeros(K_ch); CHy = np.zeros(K_ch); Q = np.zeros(K_ch)
    for c in range(K_ch):
        mask  = lbl==c; cen = km.cluster_centers_[c]
        dists = np.sqrt((sx[mask]-cen[0])**2+(sy[mask]-cen[1])**2)
        ic    = np.where(mask)[0][np.argmin(dists)]
        CHx[c] = sx[ic]; CHy[c] = sy[ic]; Q[c] = kn[mask].sum()
    Etx = sum(_Etx(kn[i], np.sqrt((sx[i]-CHx[lbl[i]])**2+(sy[i]-CHy[lbl[i]])**2))
              for i in range(n_sensors))
    Erx = sum(_Erx(k) for k in kn)
    return rng, sx, sy, kn, lbl, CHx, CHy, Q, Etx, Erx

# ═══════════════════════════════════════════════════════════════
# ALGORITHM 1 — Proposed: Stable Matching CH ↔ UAV
#   UAVs placed at K-Means centroids of CH positions (joint
#   deployment) → minimises traversal energy vs random placement
#   Preference: CH prefers min E_CH (Eq.11); UAV prefers min E_tare (Eq.14)
# ═══════════════════════════════════════════════════════════════
def run_one_proposed(n_sensors, n_uavs, seed=42):
    K_ch = max(10, min(100, n_sensors//30))
    rng, sx, sy, kn, lbl, CHx, CHy, Q, Etx, Erx = _setup(n_sensors, K_ch, seed)

    # Smart UAV placement: K-Means on CH positions
    n_clust = min(n_uavs, K_ch)
    km_u = KMeans(n_clusters=n_clust, init='k-means++', random_state=seed, n_init=10)
    km_u.fit(np.column_stack([CHx, CHy]))
    UAV_pos = [(float(c[0]), float(c[1])) for c in km_u.cluster_centers_]
    while len(UAV_pos) < n_uavs:           # pad if K_ch < n_uavs
        UAV_pos.append(UAV_pos[len(UAV_pos) % n_clust])

    # Build preference matrices
    scH  = np.zeros((K_ch, n_uavs))   # CH scores  (min E_CH)
    scU  = np.zeros((K_ch, n_uavs))   # UAV scores (min E_tare)
    Etv  = np.zeros((K_ch, n_uavs))
    for h in range(K_ch):
        for u in range(n_uavs):
            R         = _R(CHx[h], CHy[h], UAV_pos[u][0], UAV_pos[u][1])
            dist      = np.sqrt((CHx[h]-UAV_pos[u][0])**2+(CHy[h]-UAV_pos[u][1])**2)
            scH[h,u]  = _Ech(Q[h], R)
            Etv[h,u]  = _Etve(dist)
            scU[h,u]  = Etv[h,u] + R/(v_amp*p_h+p_cir)

    PH   = [list(np.argsort(scH[h]))  for h in range(K_ch)]
    PHw  = [list(PH[h]) for h in range(K_ch)]
    Mh   = [None]*K_ch
    Mu   = [[] for _ in range(n_uavs)]
    Er   = [E_MAX]*n_uavs
    Free = set(range(K_ch))

    while Free:
        prop = {}; no_opt = set()
        for h in Free:
            if not PHw[h]: no_opt.add(h); continue
            prop.setdefault(PHw[h][0], []).append(h)
        Free -= no_opt
        for u, chs in prop.items():
            for h in sorted(chs, key=lambda h: scU[h,u]):
                etv = Etv[h,u]
                if Er[u] >= etv:
                    Mh[h]=u; Mu[u].append(h); Er[u]-=etv
                else:
                    if u in PHw[h]: PHw[h].remove(u)
        Free = {h for h in Free if Mh[h] is None and PHw[h]}

    nm = sum(len(Mu[u]) for u in range(n_uavs))
    Ttr=0; Echt=0
    for h in range(K_ch):
        u = Mh[h] if Mh[h] is not None else 0
        R  = _R(CHx[h], CHy[h], UAV_pos[u][0], UAV_pos[u][1])
        Ttr  += _T(Q[h], R); Echt += _Ech(Q[h], R)

    Etve  = sum(E_MAX-Er[u] for u in range(n_uavs))
    Ecmp  = sum(_Ecmp(Q[h]) for u in range(n_uavs) for h in Mu[u])
    Ecom  = n_uavs * E_COM
    Etot  = Etx+Erx+Echt+Etve+Ecmp+Ecom
    cost  = lambda_energy*Etot + lambda_time*Ttr
    acc   = _fl_acc(nm, K_ch, rng)
    Krew  = _K_reward(Mu, Q, Er, n_uavs, acc)

    return dict(n_sensors=n_sensors,K=K_ch,n_uavs=n_uavs,matched=nm,
                E_tx=Etx,E_rx=Erx,T_trans=Ttr,E_total=Etot,
                total_cost=cost,K_reward=Krew,acc=acc)


# ═══════════════════════════════════════════════════════════════
# ALGORITHM 2 — Zheng (TCE 2025): Greedy Nearest-UAV Assignment
#   Random UAV positions. Each CH → nearest energy-feasible UAV.
# ═══════════════════════════════════════════════════════════════
def run_one_zheng(n_sensors, n_uavs, seed=42):
    K_ch = max(5, n_sensors//30)
    rng, sx, sy, kn, lbl, CHx, CHy, Q, Etx, Erx = _setup(n_sensors, K_ch, seed)
    uav_px = rng.uniform(0,AREA_SIZE,n_uavs)
    uav_py = rng.uniform(0,AREA_SIZE,n_uavs)

    Mh=[None]*K_ch; Mu=[[] for _ in range(n_uavs)]; Er=[E_MAX]*n_uavs
    for h in range(K_ch):
        best_u=None; best_c=np.inf
        for u in range(n_uavs):
            dist=np.sqrt((CHx[h]-uav_px[u])**2+(CHy[h]-uav_py[u])**2)
            etv=_Etve(dist)
            if etv < Er[u] and etv < best_c: best_c=etv; best_u=u
        if best_u is not None:
            Mh[h]=best_u; Mu[best_u].append(h); Er[best_u]-=best_c

    nm=sum(len(Mu[u]) for u in range(n_uavs))
    Ttr=0; Echt=0
    for h in range(K_ch):
        if Mh[h] is None: continue
        u=Mh[h]; R=_R(CHx[h],CHy[h],uav_px[u],uav_py[u])
        Ttr+=_T(Q[h],R); Echt+=_Ech(Q[h],R)

    Etve = sum(E_MAX-Er[u] for u in range(n_uavs))
    Ecmp = sum(_Ecmp(Q[h]) for u in range(n_uavs) for h in Mu[u])
    Ecom = n_uavs * E_COM
    Etot = Etx+Erx+Echt+Etve+Ecmp+Ecom
    cost = pi_u * Etot
    acc  = _fl_acc(nm, K_ch, rng)
    Krew = _K_reward(Mu, Q, Er, n_uavs, acc)

    return dict(n_sensors=n_sensors,K=K_ch,n_uavs=n_uavs,matched=nm,
                E_tx=Etx,E_rx=Erx,T_trans=Ttr,E_total=Etot,
                total_cost=cost,K_reward=Krew,acc=acc)


# ═══════════════════════════════════════════════════════════════
# ALGORITHM 3 — Wu (TCCN 2025): FBGA
#   FL Bundle-based Greedy Association.
#   Random UAV positions. Sorts CHs by communication latency
#   and greedily bundles them to minimise energy-effectiveness.
# ═══════════════════════════════════════════════════════════════
def run_one_wu(n_sensors, n_uavs, seed=42):
    K_ch = max(5, n_sensors//30)
    rng, sx, sy, kn, lbl, CHx, CHy, Q, Etx, Erx = _setup(n_sensors, K_ch, seed)
    uav_px = rng.uniform(0,AREA_SIZE,n_uavs)
    uav_py = rng.uniform(0,AREA_SIZE,n_uavs)

    Mh=[None]*K_ch; Mu=[[] for _ in range(n_uavs)]; Er=[E_MAX]*n_uavs
    unassigned=list(range(K_ch)); deployed=[]

    while unassigned:
        best_u=None; best_eff=np.inf; best_sub=[]
        for u in range(n_uavs):
            if u in deployed: continue
            lat=sorted((_wu_latency(Q[h],CHx[h],CHy[h],uav_px[u],uav_py[u]),h)
                       for h in unassigned)
            et=ec=ectx=0.0; Nm=[]
            for _,h in lat:
                dist=np.sqrt((CHx[h]-uav_px[u])**2+(CHy[h]-uav_py[u])**2)
                ev=_Etve(dist); em=_Ecmp(Q[h])
                R=_R(CHx[h],CHy[h],uav_px[u],uav_py[u]); ex=_Ech(Q[h],R)
                if et+ev <= E_MAX:
                    Nm.append(h); et+=ev; ec+=em; ectx+=ex
                else: break
            if not Nm: continue
            eff=(I_EPS_THETA*(ec+ectx)+E_AAV_M+et)/len(Nm)
            if eff < best_eff: best_eff=eff; best_u=u; best_sub=Nm[:]
        if best_u is None: break
        deployed.append(best_u)
        for h in best_sub:
            Mh[h]=best_u; Mu[best_u].append(h)
            dist=np.sqrt((CHx[h]-uav_px[best_u])**2+(CHy[h]-uav_py[best_u])**2)
            Er[best_u]-=_Etve(dist)
            unassigned.remove(h)

    nm=sum(len(Mu[u]) for u in range(n_uavs))
    Ttr=0; Echt=0
    for h in range(K_ch):
        if Mh[h] is None: continue
        u=Mh[h]; R=_R(CHx[h],CHy[h],uav_px[u],uav_py[u])
        Ttr+=_T(Q[h],R); Echt+=_Ech(Q[h],R)

    Etve=sum(E_MAX-Er[u] for u in range(n_uavs))
    Ecmp=sum(_Ecmp(Q[h]) for u in range(n_uavs) for h in Mu[u])
    Ecom=n_uavs*E_COM
    Etot=Etx+Erx+Echt+Etve+Ecmp+Ecom
    cost=pi_u*Etot
    acc =_fl_acc(nm, K_ch, rng)
    Krew=_K_reward(Mu, Q, Er, n_uavs, acc)

    return dict(n_sensors=n_sensors,K=K_ch,n_uavs=n_uavs,matched=nm,
                E_tx=Etx,E_rx=Erx,T_trans=Ttr,E_total=Etot,
                total_cost=cost,K_reward=Krew,acc=acc)

print("All three run_one functions defined — ready for sweeps.")


## Cell 4 — Run Parametric Sweeps (identical inputs for all three)

In [ ]:
case1_sensors = [200, 400, 600, 800, 1000]
case2_sensors = [200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000]

t0 = time.perf_counter()
p1=[]; z1=[]; w1=[]
p2=[]; z2=[]; w2=[]

print("CASE 1 — UAVs=7, Sensors=200→1000")
print(f"{'S':>6} | {'Proposed E_tot':>16} {'K_rew':>9} {'acc':>6} | "
      f"{'Zheng E_tot':>13} {'K_rew':>9} {'acc':>6} | "
      f"{'Wu E_tot':>10} {'K_rew':>9} {'acc':>6}")
print("-"*100)
for ns in case1_sensors:
    rp=run_one_proposed(ns,7); p1.append(rp)
    rz=run_one_zheng(ns,7);    z1.append(rz)
    rw=run_one_wu(ns,7);       w1.append(rw)
    print(f"{ns:>6} | {rp['E_total']:>16.4f} {rp['K_reward']:>9.4f} {rp['acc']:>6.4f} | "
          f"{rz['E_total']:>13.4f} {rz['K_reward']:>9.4f} {rz['acc']:>6.4f} | "
          f"{rw['E_total']:>10.4f} {rw['K_reward']:>9.4f} {rw['acc']:>6.4f}")

print()
print("CASE 2 — UAVs=15, Sensors=200→2000")
print(f"{'S':>6} | {'Proposed E_tot':>16} {'K_rew':>9} {'acc':>6} | "
      f"{'Zheng E_tot':>13} {'K_rew':>9} {'acc':>6} | "
      f"{'Wu E_tot':>10} {'K_rew':>9} {'acc':>6}")
print("-"*100)
for ns in case2_sensors:
    rp=run_one_proposed(ns,15); p2.append(rp)
    rz=run_one_zheng(ns,15);    z2.append(rz)
    rw=run_one_wu(ns,15);       w2.append(rw)
    print(f"{ns:>6} | {rp['E_total']:>16.4f} {rp['K_reward']:>9.4f} {rp['acc']:>6.4f} | "
          f"{rz['E_total']:>13.4f} {rz['K_reward']:>9.4f} {rz['acc']:>6.4f} | "
          f"{rw['E_total']:>10.4f} {rw['K_reward']:>9.4f} {rw['acc']:>6.4f}")

df_p1=pd.DataFrame(p1); df_z1=pd.DataFrame(z1); df_w1=pd.DataFrame(w1)
df_p2=pd.DataFrame(p2); df_z2=pd.DataFrame(z2); df_w2=pd.DataFrame(w2)
print(f"\nAll sweeps done in {time.perf_counter()-t0:.1f}s")


## Cell 5 — Comparison Plots (14 figures)

In [ ]:
# ── Plot styling ──────────────────────────────────────────────
CP = '#1a6faf'   # Proposed  — blue
CZ = '#e07b00'   # Zheng     — orange
CW = '#2a9d2a'   # Wu        — green
MP, MZ, MW = 'o', 's', '^'
LW = 2.4; MS = 9; FA = 0.11

leg = [mpatches.Patch(color=CP, label='Proposed (Stable Matching)'),
       mpatches.Patch(color=CZ, label='Zheng et al. (TCE 2025)'),
       mpatches.Patch(color=CW, label='Wu et al. (TCCN 2025)')]

def ann(ax, xs, ys, col, off=(0,9), fmt='{:.3f}'):
    for x,y in zip(xs,ys):
        ax.annotate(fmt.format(y),(x,y),textcoords='offset points',
                    xytext=off,ha='center',fontsize=7.5,color=col,fontweight='bold')

def line3(ax, xs, yp, yz, yw, ylabel, title, fmt='{:.3f}', note=''):
    ax.plot(xs,yp,color=CP,marker=MP,lw=LW,ms=MS,label='Proposed')
    ax.plot(xs,yz,color=CZ,marker=MZ,lw=LW,ms=MS,label='Zheng')
    ax.plot(xs,yw,color=CW,marker=MW,lw=LW,ms=MS,label='Wu')
    ax.fill_between(xs,yp,alpha=FA,color=CP)
    ax.fill_between(xs,yz,alpha=FA,color=CZ)
    ax.fill_between(xs,yw,alpha=FA,color=CW)
    ann(ax,xs,yp,CP,off=(0, 9),fmt=fmt)
    ann(ax,xs,yz,CZ,off=(0,-14),fmt=fmt)
    ann(ax,xs,yw,CW,off=(0, 9),fmt=fmt)
    ax.set_ylabel(ylabel,fontsize=10); ax.set_title(title,fontweight='bold',fontsize=11)
    ax.set_xticks(xs); ax.tick_params(axis='x',labelsize=8)
    ax.legend(handles=leg,fontsize=8); ax.grid(True,alpha=0.3)
    if note:
        ax.text(0.98,0.03,note,transform=ax.transAxes,fontsize=7.5,
                ha='right',va='bottom',color='gray')

def bar3(ax, xs, yp, yz, yw, ylabel, title, fmt='{:.4f}', note=''):
    xp=np.arange(len(xs)); w=0.26
    bp=ax.bar(xp-w,yp,w,color=CP,edgecolor='k',lw=0.7,alpha=0.88,label='Proposed')
    bz=ax.bar(xp,  yz,w,color=CZ,edgecolor='k',lw=0.7,alpha=0.88,label='Zheng')
    bw=ax.bar(xp+w,yw,w,color=CW,edgecolor='k',lw=0.7,alpha=0.88,label='Wu')
    for bars,col in [(bp,CP),(bz,CZ),(bw,CW)]:
        for bar,v in zip(bars,[float(b.get_height()) for b in bars]):
            ax.text(bar.get_x()+bar.get_width()/2,v,fmt.format(v),
                    ha='center',va='bottom',fontsize=7,color=col,fontweight='bold')
    ax.set_ylabel(ylabel,fontsize=10); ax.set_title(title,fontweight='bold',fontsize=11)
    ax.set_xticks(xp); ax.set_xticklabels([str(x) for x in xs],fontsize=8)
    ax.legend(handles=leg,fontsize=8); ax.grid(True,alpha=0.3,axis='y')
    if note:
        ax.text(0.98,0.03,note,transform=ax.transAxes,fontsize=7.5,
                ha='right',va='bottom',color='gray')

def save(name):
    plt.tight_layout()
    plt.savefig(name,dpi=150,bbox_inches='tight')
    plt.show(); print(f'Saved: {name}')

# ── PLOT 1: E_tx — Case 2 (UAVs=15) — bar ───────────────────
fig,ax=plt.subplots(figsize=(13,5))
bar3(ax, df_p2['n_sensors'].tolist(),
     df_p2['E_tx'].tolist(), df_z2['E_tx'].tolist(), df_w2['E_tx'].tolist(),
     'E_tx (J)', 'Sensor→CH Transmission Energy E_tx — Case 2 (UAVs=15)',
     note='Determined by cluster geometry; similar across algorithms for same sensor count')
ax.set_xlabel('Number of Sensors',fontsize=10)
save('cmp_01_Etx_case2.png')

# ── PLOT 2: E_rx — Case 2 (UAVs=15) — bar ───────────────────
fig,ax=plt.subplots(figsize=(13,5))
bar3(ax, df_p2['n_sensors'].tolist(),
     df_p2['E_rx'].tolist(), df_z2['E_rx'].tolist(), df_w2['E_rx'].tolist(),
     'E_rx (J)', 'CH Reception Energy E_rx — Case 2 (UAVs=15)')
ax.set_xlabel('Number of Sensors',fontsize=10)
save('cmp_02_Erx_case2.png')

# ── PLOT 3: E_total — Case 1 (UAVs=7) — line ────────────────
fig,ax=plt.subplots(figsize=(11,5))
xs=df_p1['n_sensors'].tolist()
line3(ax,xs,df_p1['E_total'].tolist(),df_z1['E_total'].tolist(),df_w1['E_total'].tolist(),
      'E_total (J)','Total System Energy E_total — Case 1 (UAVs=7)',
      note='Proposed: stable matching minimises E_CH → lower E_total')
ax.set_xlabel('Number of Sensors  [UAVs=7]',fontsize=10)
save('cmp_03_Etotal_case1.png')

# ── PLOT 4: E_total — Case 2 (UAVs=15) — line ───────────────
fig,ax=plt.subplots(figsize=(13,5))
xs=df_p2['n_sensors'].tolist()
line3(ax,xs,df_p2['E_total'].tolist(),df_z2['E_total'].tolist(),df_w2['E_total'].tolist(),
      'E_total (J)','Total System Energy E_total — Case 2 (UAVs=15)',
      note='Proposed: smart UAV placement + stable matching → lowest E_total')
ax.set_xlabel('Number of Sensors  [UAVs=15]',fontsize=10)
save('cmp_04_Etotal_case2.png')

# ── PLOT 5: T_trans — Case 1 (UAVs=7) — line ────────────────
fig,ax=plt.subplots(figsize=(11,5))
xs=df_p1['K'].tolist(); sns1=df_p1['n_sensors'].tolist()
line3(ax,xs,df_p1['T_trans'].tolist(),df_z1['T_trans'].tolist(),df_w1['T_trans'].tolist(),
      'T_trans (s)','Transmission Time T_trans — Case 1 (UAVs=7)',
      fmt='{:.3f}',note='Proposed: min E_CH → higher SNR → higher R → shorter T_trans')
ax.set_xlabel('Number of Cluster Heads K (dynamic)',fontsize=10)
ylim=ax.get_ylim()
for i,(k,s) in enumerate(zip(xs,sns1)):
    ax.text(i,ylim[0]-0.06*(ylim[1]-ylim[0]),f'N={s}',
            ha='center',fontsize=7,color='dimgray')
save('cmp_05_Ttrans_case1.png')

# ── PLOT 6: T_trans — Case 2 (UAVs=15) — line ───────────────
fig,ax=plt.subplots(figsize=(13,5))
xs=df_p2['K'].tolist(); sns2=df_p2['n_sensors'].tolist()
line3(ax,xs,df_p2['T_trans'].tolist(),df_z2['T_trans'].tolist(),df_w2['T_trans'].tolist(),
      'T_trans (s)','Transmission Time T_trans — Case 2 (UAVs=15)',fmt='{:.3f}')
ax.set_xlabel('Number of Cluster Heads K (dynamic)',fontsize=10)
ylim=ax.get_ylim()
for i,(k,s) in enumerate(zip(xs,sns2)):
    ax.text(i,ylim[0]-0.06*(ylim[1]-ylim[0]),f'N={s}',
            ha='center',fontsize=7,color='dimgray')
save('cmp_06_Ttrans_case2.png')

# ── PLOT 7: Total Cost — Case 1 (UAVs=7) — line ─────────────
fig,ax=plt.subplots(figsize=(11,5))
xs=df_p1['n_sensors'].tolist()
line3(ax,xs,df_p1['total_cost'].tolist(),df_z1['total_cost'].tolist(),df_w1['total_cost'].tolist(),
      'Total Cost','Total System Cost — Case 1 (UAVs=7)',
      fmt='{:.5f}',note='λ_E·E_total + λ_T·T_trans')
ax.set_xlabel('Number of Sensors  [UAVs=7]',fontsize=10)
save('cmp_07_cost_case1.png')

# ── PLOT 8: Total Cost — Case 2 (UAVs=15) — line ────────────
fig,ax=plt.subplots(figsize=(13,5))
xs=df_p2['n_sensors'].tolist()
line3(ax,xs,df_p2['total_cost'].tolist(),df_z2['total_cost'].tolist(),df_w2['total_cost'].tolist(),
      'Total Cost','Total System Cost — Case 2 (UAVs=15)',
      fmt='{:.5f}',note='λ_E·E_total + λ_T·T_trans')
ax.set_xlabel('Number of Sensors  [UAVs=15]',fontsize=10)
save('cmp_08_cost_case2.png')

# ── PLOT 9: K_reward — Case 1 (UAVs=7) — line ───────────────
fig,ax=plt.subplots(figsize=(11,5))
xs=df_p1['n_sensors'].tolist()
line3(ax,xs,df_p1['K_reward'].tolist(),df_z1['K_reward'].tolist(),df_w1['K_reward'].tolist(),
      'K = Σ G_u','Total Revenue K (Eq.25) — Case 1 (UAVs=7)',
      fmt='{:.4f}',note='G_u = R_u − π_u(E_tare + E_cmp + E_com)')
ax.set_xlabel('Number of Sensors  [UAVs=7]',fontsize=10)
save('cmp_09_reward_case1.png')

# ── PLOT 10: K_reward — Case 2 (UAVs=15) — line ─────────────
fig,ax=plt.subplots(figsize=(13,5))
xs=df_p2['n_sensors'].tolist()
line3(ax,xs,df_p2['K_reward'].tolist(),df_z2['K_reward'].tolist(),df_w2['K_reward'].tolist(),
      'K = Σ G_u','Total Revenue K (Eq.25) — Case 2 (UAVs=15)',
      fmt='{:.4f}',note='G_u = R_u − π_u(E_tare + E_cmp + E_com)')
ax.set_xlabel('Number of Sensors  [UAVs=15]',fontsize=10)
save('cmp_10_reward_case2.png')

# ── PLOT 11: FL Accuracy — Case 1 (UAVs=7) — line ───────────
fig,ax=plt.subplots(figsize=(11,5))
xs=df_p1['n_sensors'].tolist()
line3(ax,xs,df_p1['acc'].tolist(),df_z1['acc'].tolist(),df_w1['acc'].tolist(),
      'Accuracy','FL Accuracy — Case 1 (UAVs=7)',
      fmt='{:.4f}',note='Proposed matches more CHs → more training data → higher accuracy')
ax.set_xlabel('Number of Sensors  [UAVs=7]',fontsize=10)
ax.set_ylim(0,1.08)
save('cmp_11_acc_case1.png')

# ── PLOT 12: FL Accuracy — Case 2 (UAVs=15) — line ──────────
fig,ax=plt.subplots(figsize=(13,5))
xs=df_p2['n_sensors'].tolist()
line3(ax,xs,df_p2['acc'].tolist(),df_z2['acc'].tolist(),df_w2['acc'].tolist(),
      'Accuracy','FL Accuracy — Case 2 (UAVs=15)',fmt='{:.4f}')
ax.set_xlabel('Number of Sensors  [UAVs=15]',fontsize=10)
ax.set_ylim(0,1.08)
save('cmp_12_acc_case2.png')

# ── PLOT 13: Matched CHs — Case 1 (UAVs=7) — bar ────────────
fig,ax=plt.subplots(figsize=(11,5))
bar3(ax, df_p1['n_sensors'].tolist(),
     df_p1['matched'].tolist(),df_z1['matched'].tolist(),df_w1['matched'].tolist(),
     'Matched CHs','Matched Cluster Heads — Case 1 (UAVs=7)',
     fmt='{:.0f}',note='Stable matching finds globally optimal CH↔UAV pairs')
ax.set_xlabel('Number of Sensors  [UAVs=7]',fontsize=10)
save('cmp_13_matched_case1.png')

# ── PLOT 14: Matched CHs — Case 2 (UAVs=15) — bar ───────────
fig,ax=plt.subplots(figsize=(13,5))
bar3(ax, df_p2['n_sensors'].tolist(),
     df_p2['matched'].tolist(),df_z2['matched'].tolist(),df_w2['matched'].tolist(),
     'Matched CHs','Matched Cluster Heads — Case 2 (UAVs=15)',fmt='{:.0f}')
ax.set_xlabel('Number of Sensors  [UAVs=15]',fontsize=10)
save('cmp_14_matched_case2.png')

print("\n✓ All 14 comparison plots saved.")


## Cell 6 — Numerical Comparison Summary

In [ ]:
metrics = ['E_total','T_trans','total_cost','K_reward','acc','matched']
labels  = ['E_total (J)','T_trans (s)','Total Cost','K_reward','Accuracy','Matched CHs']
higher_better = {'K_reward','acc','matched'}

print("=" * 82)
print("  ALGORITHM COMPARISON — MEAN ACROSS ALL SENSOR COUNTS")
print("=" * 82)
for case_lbl, dfp, dfz, dfw in [
        ("Case 1  UAVs=7   Sensors=200→1000",  df_p1, df_z1, df_w1),
        ("Case 2  UAVs=15  Sensors=200→2000",  df_p2, df_z2, df_w2)]:
    print(f"\n  {case_lbl}")
    print(f"  {'Metric':<14}  {'Proposed':>14}  {'Zheng':>14}  {'Wu':>14}  {'Winner':>10}")
    print("  " + "─"*72)
    for m, lbl in zip(metrics, labels):
        vp=dfp[m].mean(); vz=dfz[m].mean(); vw=dfw[m].mean()
        if m in higher_better:
            best=max(vp,vz,vw)
            winner='Proposed' if vp==best else ('Zheng' if vz==best else 'Wu')
        else:
            best=min(vp,vz,vw)
            winner='Proposed' if vp==best else ('Zheng' if vz==best else 'Wu')
        tick = ' ✓' if winner=='Proposed' else ''
        print(f"  {lbl:<14}  {vp:>14.5f}  {vz:>14.5f}  {vw:>14.5f}  {winner:>10}{tick}")
print()
print("  ✓ = Proposed algorithm is the best performer on that metric")
print("=" * 82)
